<a href="https://colab.research.google.com/github/Sajani-Prabhashika/SDGP/blob/main/Test02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
import os
import kagglehub

# --- STEP 1: DOWNLOAD DATASET ---
print("Downloading dataset from Kaggle...")
# This downloads the files and returns the path where they are stored
path = kagglehub.dataset_download("madhavipethangoda/cinnamon-plant-stem-and-branch-disease-dataset")

print(f"Dataset downloaded to: {path}")

# Check what's inside the folder to make sure we point to the right place
print("Contents of dataset folder:", os.listdir(path))

# usually kagglehub downloads to a folder. We set that as our data source.
DATA_DIR = path

# --- CONFIGURATION ---
BATCH_SIZE = 32
IMG_SIZE = (224, 224)
EPOCHS = 15

# --- STEP 2: PREPARE DATA ---
# Data Augmentation (Crucial for small datasets)
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2  # 20% for testing
)

print("Loading Training Data...")
try:
    train_generator = train_datagen.flow_from_directory(
        DATA_DIR,
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        subset='training'
    )

    print("Loading Validation Data...")
    validation_generator = train_datagen.flow_from_directory(
        DATA_DIR,
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        subset='validation'
    )
except:
    print("\n ERROR: Could not find images directly in the path.")
    print("Please check the 'Contents of dataset folder' printed above.")
    print("You might need to add the subfolder name to DATA_DIR (e.g., path + '/dataset')\n")
    raise

# Print the class labels to confirm (e.g., Rough Bark, Stripe Canker)
print("Classes found:", train_generator.class_indices)

# --- STEP 3: BUILD MODEL (MobileNetV2) ---
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=IMG_SIZE + (3,))
base_model.trainable = False  # Freeze base layers

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.2)(x)
predictions = Dense(train_generator.num_classes, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)

model.compile(optimizer=Adam(learning_rate=0.0001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# --- STEP 4: TRAIN ---
print("Starting Training...")
history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=validation_generator
)

# --- STEP 5: SAVE AS TFLITE ---
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

# Save the model file
with open('teera_model.tflite', 'wb') as f:
    f.write(tflite_model)

# Save the labels file
print("Saving Labels...")
with open('labels.txt', 'w') as f:
    for class_name in list(train_generator.class_indices.keys()):
        f.write(class_name + '\n')

print("\n SUCCESS!")
print("1. Download 'teera_model.tflite'")
print("2. Download 'labels.txt'")

100%|██████████| 664M/664M [00:06<00:00, 106MB/s]

Extracting files...


Dataset downloaded to: /root/.cache/kagglehub/datasets/madhavipethangoda/cinnamon-plant-stem-and-branch-disease-dataset/versions/1
Contents of dataset folder: ['Diseases']
Loading Training Data...
Found 261 images belonging to 1 classes.
Loading Validation Data...
Found 65 images belonging to 1 classes.
✅ Classes found: {'Diseases': 0}
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Starting Training...


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/15


/usr/local/lib/python3.12/dist-packages/keras/src/ops/nn.py:944: UserWarning: You are using a softmax over axis -1 of a tensor of shape (None, 1). This axis has size 1. The softmax operation will always return the value 1, which is likely not what you intended. Did you mean to use a sigmoid instead?
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/keras/src/losses/losses.py:33: SyntaxWarning: In loss categorical_crossentropy, expected y_pred.shape to be (batch_size, num_classes) with num_classes > 1. Received: y_pred.shape=(None, 1). Consider using 'binary_crossentropy' if you only have 2 classes.
  return self.fn(y_true, y_pred, **self._fn_kwargs)


9/9 ━━━━━━━━━━━━━━━━━━━━ 55s 5s/step - accuracy: 1.0000 - loss: 0.0000e+00 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 2/15
9/9 ━━━━━━━━━━━━━━━━━━━━ 44s 5s/step - accuracy: 1.0000 - loss: 0.0000e+00 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 3/15
9/9 ━━━━━━━━━━━━━━━━━━━━ 41s 4s/step - accuracy: 1.0000 - loss: 0.0000e+00 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 4/15
9/9 ━━━━━━━━━━━━━━━━━━━━ 41s 4s/step - accuracy: 1.0000 - loss: 0.0000e+00 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 5/15
9/9 ━━━━━━━━━━━━━━━━━━━━ 40s 4s/step - accuracy: 1.0000 - loss: 0.0000e+00 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 6/15
9/9 ━━━━━━━━━━━━━━━━━━━━ 40s 4s/step - accuracy: 1.0000 - loss: 0.0000e+00 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 7/15
9/9 ━━━━━━━━━━━━━━━━━━━━ 41s 4s/step - accuracy: 1.0000 - loss: 0.0000e+00 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 8/15
9/9 ━━━━━━━━━━━━━━━━━━━━ 39s 4s/step - accuracy: 1.0000 - loss: 0.0000e+00 -

In [1]:
import os
from google.colab import files

# 1. Search for the files in the entire system
found = False
print("Searching for files...")

for root, dirs, filenames in os.walk("/content/"):
    for filename in filenames:
        if filename == "teera_model.tflite" or filename == "labels.txt":
            full_path = os.path.join(root, filename)
            print(f"FOUND: {full_path}")
            found = True
            # Attempt to trigger download automatically
            files.download(full_path)

if not found:
    print("Files were NOT found. The training probably crashed.")
    print("Please scroll up to the previous code cell and check for RED error messages.")

Searching for files...
Files were NOT found. The training probably crashed.
Please scroll up to the previous code cell and check for RED error messages.


In [2]:
import os

# Check if the AI model still exists in the memory
if 'tflite_model' in globals():
    print("✅ Model found in memory! Saving to main folder...")

    # Force save to /content/ (The main folder)
    with open('/content/teera_model.tflite', 'wb') as f:
        f.write(tflite_model)

    # Force save labels
    if 'train_generator' in globals():
        with open('/content/labels.txt', 'w') as f:
            for class_name in list(train_generator.class_indices.keys()):
                f.write(class_name + '\n')
        print("✅ Labels saved!")
    else:
        print("⚠️ Labels data missing (but model is safe).")

    print("\n👇 ACTION REQUIRED:")
    print("1. Go to the file browser on the left.")
    print("2. Click the REFRESH folder icon (top of the panel).")
    print("3. Look directly under 'sample_data' (not inside it).")

else:
    print("❌ The model is NOT in memory.")
    print("This means the session restarted. You MUST run the Training Code block again.")

❌ The model is NOT in memory.
This means the session restarted. You MUST run the Training Code block again.


In [3]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
import os
import kagglehub
from google.colab import files
import shutil

# --- STEP 1: DOWNLOAD DATASET ---
print("⬇️ Downloading dataset from Kaggle...")
path = kagglehub.dataset_download("madhavipethangoda/cinnamon-plant-stem-and-branch-disease-dataset")
print(f"✅ Dataset downloaded to: {path}")

# --- CONFIGURATION ---
BATCH_SIZE = 32
IMG_SIZE = (224, 224)
EPOCHS = 15

# --- STEP 2: PREPARE DATA ---
print("⚙️ Setting up Data Generators...")

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2
)

# Robustly find the correct folder
# Sometimes kagglehub puts files inside a subfolder. We check for that.
target_dir = path
if 'dataset' in os.listdir(path):
    target_dir = os.path.join(path, 'dataset')

print(f"📂 Reading images from: {target_dir}")

train_generator = train_datagen.flow_from_directory(
    target_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'
)

validation_generator = train_datagen.flow_from_directory(
    target_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation'
)

print("🏷️ Classes detected:", train_generator.class_indices)

# --- STEP 3: BUILD MODEL (MobileNetV2) ---
print("🧠 Building MobileNetV2 Model...")
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=IMG_SIZE + (3,))
base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.2)(x)
predictions = Dense(train_generator.num_classes, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)

model.compile(optimizer=Adam(learning_rate=0.0001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

# --- STEP 4: TRAIN ---
print("🚀 Starting Training (This will take a few minutes)...")
history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=validation_generator
)

# --- STEP 5: SAVE & CONVERT ---
print("💾 Saving Model...")
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

# Save to the main /content/ folder so it's easy to see
tflite_path = '/content/teera_model.tflite'
labels_path = '/content/labels.txt'

with open(tflite_path, 'wb') as f:
    f.write(tflite_model)

with open(labels_path, 'w') as f:
    for class_name in list(train_generator.class_indices.keys()):
        f.write(class_name + '\n')

print("✅ Files saved to /content/ folder!")

# --- STEP 6: AUTO-DOWNLOAD ---
print("⬇️ Attempting automatic download...")
try:
    files.download(tflite_path)
    files.download(labels_path)
except Exception as e:
    print("⚠️ Auto-download blocked. Please download manually from the file browser on the left.")

⬇️ Downloading dataset from Kaggle...


100%|██████████| 664M/664M [00:08<00:00, 80.8MB/s]

Extracting files...


✅ Dataset downloaded to: /root/.cache/kagglehub/datasets/madhavipethangoda/cinnamon-plant-stem-and-branch-disease-dataset/versions/1
⚙️ Setting up Data Generators...
📂 Reading images from: /root/.cache/kagglehub/datasets/madhavipethangoda/cinnamon-plant-stem-and-branch-disease-dataset/versions/1
Found 261 images belonging to 1 classes.
Found 65 images belonging to 1 classes.
🏷️ Classes detected: {'Diseases': 0}
🧠 Building MobileNetV2 Model...
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
🚀 Starting Training (This will take a few minutes)...


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


...

9/9 ━━━━━━━━━━━━━━━━━━━━ 52s 5s/step - accuracy: 1.0000 - loss: 0.0000e+00 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 2/15
9/9 ━━━━━━━━━━━━━━━━━━━━ 38s 4s/step - accuracy: 1.0000 - loss: 0.0000e+00 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 3/15
9/9 ━━━━━━━━━━━━━━━━━━━━ 38s 4s/step - accuracy: 1.0000 - loss: 0.0000e+00 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 4/15
9/9 ━━━━━━━━━━━━━━━━━━━━ 38s 4s/step - accuracy: 1.0000 - loss: 0.0000e+00 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 5/15
9/9 ━━━━━━━━━━━━━━━━━━━━ 38s 4s/step - accuracy: 1.0000 - loss: 0.0000e+00 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 6/15
9/9 ━━━━━━━━━━━━━━━━━━━━ 38s 4s/step - accuracy: 1.0000 - loss: 0.0000e+00 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 7/15
9/9 ━━━━━━━━━━━━━━━━━━━━ 37s 4s/step - accuracy: 1.0000 - loss: 0.0000e+00 - val_accuracy: 1.0000 - val_loss: 0.0000e+00
Epoch 8/15
9/9 ━━━━━━━━━━━━━━━━━━━━ 38s 4s/step - accuracy: 1.0000 - loss: 0.0000e+00 -

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>